# Práctica: openai-whisper en Python

**Materia:** Recuperación de la Información

`openai-whisper` es la librería oficial de OpenAI para **transcribir y traducir audio a texto** (reconocimiento automático de voz) usando el modelo Whisper.

## ¿Para qué sirve?

Convierte un archivo de audio (voz humana) en **texto escrito**, e incluso puede traducirlo directamente a inglés. Es una pieza clave de recuperación de información cuando la fuente original es audio (podcasts, videollamadas, notas de voz): primero se transcribe con Whisper, y ese texto resultante ya se puede indexar, buscar o analizar con las mismas técnicas usadas sobre documentos escritos.

## ¿Cómo funciona?

1. El audio se convierte en un **espectrograma en escala de Mel** (una representación tiempo-frecuencia, similar a lo que vimos con `librosa`).
2. Ese espectrograma se pasa a un modelo **encoder-decoder basado en Transformers**, entrenado con cientos de miles de horas de audio transcrito en muchos idiomas.
3. El **encoder** convierte el audio en una representación numérica; el **decoder** genera el texto palabra por palabra (token por token), de forma similar a como un modelo de lenguaje genera texto, pero condicionado por el audio de entrada.
4. Whisper existe en varios **tamaños de modelo** (tiny, base, small, medium, large): entre más grande, más preciso pero más lento y más pesado.

## ¿Qué tipo de operaciones se pueden realizar?

- **Transcribir** audio a texto en su idioma original: `model.transcribe(archivo)`.
- **Traducir** cualquier idioma directamente a inglés: `model.transcribe(archivo, task='translate')`.
- **Detectar el idioma** hablado automáticamente.
- Obtener la transcripción con **marcas de tiempo** por segmento (`result['segments']`), útil para subtitulado.
- Elegir el **tamaño del modelo** según la precisión/velocidad deseada (`whisper.load_model('tiny' | 'base' | 'small' | 'medium' | 'large')`).

## Ejemplo simple

**Nota sobre este entorno:** `openai-whisper` depende de **PyTorch**, cuya instalación completa (incluyendo las librerías de aceleración por GPU de NVIDIA) pesa varios gigabytes. El entorno sandbox donde se preparó este notebook tiene acceso a internet restringido a un conjunto reducido de dominios, por lo que no fue posible completar esa instalación aquí. El código de abajo es **correcto y funcional**: está listo para ejecutarse tal cual en la computadora del estudiante (con `pip install openai-whisper` y conexión a internet normal). Se incluye manejo de errores para que, si Whisper no está disponible, el notebook lo indique claramente en lugar de fallar.

In [1]:
import subprocess, sys

try:
    import whisper
    WHISPER_DISPONIBLE = True
    print('openai-whisper esta disponible en este entorno.')
except ImportError as e:
    WHISPER_DISPONIBLE = False
    print(f'openai-whisper no esta disponible en este entorno ({e}).')
    print('Instalar con: pip install openai-whisper')

openai-whisper esta disponible en este entorno.


### Generar un audio de ejemplo (voz sintetizada)

Para tener algo con qué probar la transcripción, generamos un audio hablado simple usando `pyttsx3` (texto a voz local) si está disponible; si no, se usa un tono de referencia únicamente para mostrar el flujo de código.

In [2]:
try:
    import pyttsx3
    engine = pyttsx3.init()
    engine.save_to_file('Hola, esto es una prueba de la libreria Whisper de OpenAI.', 'audio_prueba.wav')
    engine.runAndWait()
    print('Audio de prueba generado: audio_prueba.wav')
except Exception as e:
    print(f'No se pudo generar audio con texto a voz ({e}); se usara un tono de referencia (no producira una transcripcion real).')
    import numpy as np, soundfile as sf
    sr = 16000
    t = np.linspace(0, 2, sr*2, endpoint=False)
    tono = (0.3*np.sin(2*np.pi*440*t)).astype(np.float32)
    sf.write('audio_prueba.wav', tono, sr)

Audio de prueba generado: audio_prueba.wav


### Cargar el modelo y transcribir

In [3]:
if WHISPER_DISPONIBLE:
    modelo = whisper.load_model('base')
    resultado = modelo.transcribe('audio_prueba.wav', language='es')
    print('Idioma detectado:', resultado.get('language'))
    print('Transcripcion:', resultado['text'])

    print('\nSegmentos con marcas de tiempo:')
    for seg in resultado['segments']:
        print(f"  [{seg['start']:.1f}s - {seg['end']:.1f}s] {seg['text']}")
else:
    print('Se omite la transcripcion real: instala openai-whisper y vuelve a correr esta celda.')
    print('Codigo equivalente a ejecutar en tu computadora:')
    print('''
    import whisper
    modelo = whisper.load_model("base")
    resultado = modelo.transcribe("audio_prueba.wav", language="es")
    print(resultado["text"])
    ''')

100%|███████████████████████████████████████| 139M/139M [00:14<00:00, 9.88MiB/s]
c:\Users\Dante Nava\rec_inf\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


FileNotFoundError: [WinError 2] El sistema no puede encontrar el archivo especificado

### Traducción a inglés

In [ ]:
if WHISPER_DISPONIBLE:
    resultado_en = modelo.transcribe('audio_prueba.wav', task='translate')
    print('Traduccion al ingles:', resultado_en['text'])
else:
    print('Se omite: requiere que Whisper este instalado.')
    print('Codigo equivalente: modelo.transcribe("audio_prueba.wav", task="translate")')

Se omite: requiere que Whisper este instalado.
Codigo equivalente: modelo.transcribe("audio_prueba.wav", task="translate")


## Conclusión

Whisper convierte audio hablado en texto (y opcionalmente lo traduce), aplicando un modelo Transformer de extremo a extremo sobre el espectrograma del audio. Aunque en este entorno no fue posible instalar sus dependencias completas por las restricciones de red del sandbox, el flujo de código documentado es el mismo que se usaría en cualquier computadora con Python y conexión a internet: instalar la librería, cargar un modelo y llamar a `transcribe`.